In [76]:
# Import necessary libraries
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Data Cleaning

In [77]:
# Loading the population data
population = pd.read_csv(".\data\county_year_population.csv")

# Displaying the first few rows 
population.tail(10)

#print(population['county'])

,county,year,population,land_area_km2,density_km2,growth_pct_per_yr
1212,West Pokot,2016,586460,9123,64.3,1.92
1213,West Pokot,2017,597832,9123,65.5,1.92
1214,West Pokot,2018,609424,9123,66.8,1.92
1215,West Pokot,2019,621241,9123,68.1,1.92
1216,West Pokot,2020,633287,9123,69.4,1.92
1217,West Pokot,2021,645567,9123,70.8,1.92
1218,West Pokot,2022,658085,9123,72.1,1.92
1219,West Pokot,2023,670845,9123,73.5,1.92
1220,West Pokot,2024,683853,9123,75.0,1.92
1221,West Pokot,2025,697113,9123,76.4,1.92


In [78]:
# Loading kenya_crosswalk
crosswalk = pd.read_csv(".\data\data\kenya_county_crosswalk.csv")

# first few
crosswalk['county']

0             Mombasa
1               Kwale
2              Kilifi
3          Tana River
4                Lamu
5        Taita-Taveta
6             Garissa
7               Wajir
8             Mandera
9            Marsabit
10             Isiolo
11               Meru
12      Tharaka-Nithi
13               Embu
14              Kitui
15           Machakos
16            Makueni
17          Nyandarua
18              Nyeri
19          Kirinyaga
20           Murang'a
21             Kiambu
22            Turkana
23         West Pokot
24            Samburu
25        Trans Nzoia
26        Uasin Gishu
27    Elgeyo-Marakwet
28              Nandi
29            Baringo
30           Laikipia
31             Nakuru
32              Narok
33            Kajiado
34            Kericho
35              Bomet
36           Kakamega
37             Vihiga
38            Bungoma
39              Busia
40              Siaya
41             Kisumu
42           Homa Bay
43             Migori
44              Kisii
45        

In [79]:
# Loading land cover county
landcover = pd.read_csv(".\data\data\worldcover_landcover_county.csv")

# Isolating the percentages
calculate = landcover[['pct_bare', 'pct_bare', 'pct_crop', 'pct_grass', 'pct_mangrove', 'pct_shrub', 'pct_tree', 'pct_water', 'pct_wetland']]
landcover['total'] = calculate.select_dtypes(include="number").sum(axis=1).round(1)

# Removing total != 1.0
#landcover = landcover[landcover['total'] >= 1.0]
#landcover.head(48)

In [80]:
landcover.info()

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   system:index  47 non-null     str    
 1   COUNT         47 non-null     int64  
 2   COUNTYCO      47 non-null     int64  
 3   FIRST_COUN    47 non-null     str    
 4   area_km2      47 non-null     float64
 5   pct_bare      47 non-null     float64
 6   pct_built     47 non-null     float64
 7   pct_crop      47 non-null     float64
 8   pct_grass     47 non-null     float64
 9   pct_mangrove  47 non-null     float64
 10  pct_shrub     47 non-null     float64
 11  pct_tree      47 non-null     float64
 12  pct_water     47 non-null     float64
 13  pct_wetland   47 non-null     float64
 14  .geo          47 non-null     str    
 15  total         47 non-null     float64
dtypes: float64(11), int64(2), str(3)
memory usage: 6.0 KB


In [81]:
landcover.columns

Index(['system:index', 'COUNT', 'COUNTYCO', 'FIRST_COUN', 'area_km2',
       'pct_bare', 'pct_built', 'pct_crop', 'pct_grass', 'pct_mangrove',
       'pct_shrub', 'pct_tree', 'pct_water', 'pct_wetland', '.geo', 'total'],
      dtype='str')

In [82]:
# Loading kenya_county geo area
geography = gpd.read_file(".\data\data\kenya_counties.geojson")

In [83]:
population.info(), crosswalk.info(), landcover.info()

<class 'pandas.DataFrame'>
RangeIndex: 1222 entries, 0 to 1221
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   county             1222 non-null   str    
 1   year               1222 non-null   int64  
 2   population         1222 non-null   int64  
 3   land_area_km2      1222 non-null   int64  
 4   density_km2        1222 non-null   float64
 5   growth_pct_per_yr  1222 non-null   float64
dtypes: float64(2), int64(3), str(1)
memory usage: 57.4 KB
<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   county            47 non-null     str  
 1   knbs_code         47 non-null     int64
 2   pcode             47 non-null     str  
 3   desinventar_code  47 non-null     int64
 4   desinventar_name  47 non-null     str  
dtypes: int64(2), str(3)
memory usage: 2.0 KB
<class 'pa

(None, None, None)

In [ ]:
# Data cleaning function
def population_cleaning(data, data2, data3):
    # 1. Keeping the years from 2019
    data = data[data['year']==2019]

    # 2. Add log_density
    data['log_density'] = np.log(data['density_km2'])

    # 3. Adding county codes from crosswalk
    #data["county_codes"] = data["county"].map(data2["pcode"])
    county_to_code = data2.set_index("county")["knbs_code"]
    data["county_codes"] = data["county"].map(county_to_code)

    # 3.2 correcting the spelling of Taita Taveta
    data["county"] = data["county"].replace({"Taita Taveta": "Taita-Taveta"})
    data["county_codes"] = data["county"].map(county_to_code)

    # 3.3 Adding desinventar_code to the data
    crosswalk_by_county = data2.set_index("county")
    data["desinventar_code"] = data["county"].map(crosswalk_by_county["desinventar_code"])
    
    # 4. Reordering the positions of the columns
    data = data[['county_codes', 'county', 'desinventar_code', 'year', 'population', 'land_area_km2', 'density_km2', 'log_density', 'growth_pct_per_yr']]

    # 5.1 Joining the landcover data into the main data
    # 5.2 Creating a list of the columns to add
    country_to_landcover = ['COUNTYCO','pct_bare', 'pct_bare', 'pct_crop', 'pct_grass', 'pct_mangrove', 'pct_shrub', 'pct_tree', 'pct_water', 'pct_wetland']
    data = pd.merge(
        data,
        data3[country_to_landcover],
        left_on="county_codes",  # The column name in data
        right_on="COUNTYCO",  # The column name in data3
        how="left",
    )
    data = data.drop("COUNTYCO", axis=1)

    # Return the data
    return data

In [87]:
# Calling the function
df = population_cleaning(population, crosswalk, landcover)
df = df.sort_values(by='county_codes').reset_index(drop=True)
df.head(47)

,county_codes,county,desinventar_code,year,population,land_area_km2,density_km2,log_density,growth_pct_per_yr,pct_bare,pct_bare,pct_crop,pct_grass,pct_mangrove,pct_shrub,pct_tree,pct_water,pct_wetland
0,1,Mombasa,28,2019,1208333,220,5492.4,8.611121,2.52,0.002472,0.002472,0.048634,0.130316,0.000000,0.500581,0.290396,0.025629,4.860347e-04
1,2,Kwale,19,2019,866820,8267,104.9,4.653008,2.88,0.000138,0.000138,0.023655,0.329924,0.000000,0.291430,0.351073,0.000111,1.175836e-04
2,3,Kilifi,14,2019,1453787,12540,115.9,4.752728,2.70,0.000174,0.000174,0.363496,0.124589,0.000000,0.211135,0.290265,0.000087,1.140689e-04
3,4,Tana River,40,2019,315943,37951,8.3,2.116256,2.75,0.000402,0.000402,0.198725,0.198304,0.000000,0.409546,0.148480,0.006333,2.783485e-02
4,5,Lamu,21,2019,143920,6253,23.0,3.135494,3.49,0.001977,0.001977,0.149025,0.098885,0.000000,0.471857,0.259881,0.012694,7.423277e-04
5,6,Taita-Taveta,39,2019,340671,17152,19.9,2.990720,1.80,0.002210,0.002210,0.001662,0.179479,0.000000,0.739879,0.075553,0.000399,6.897950e-05
6,7,Garissa,6,2019,841353,44736,18.8,2.933857,3.00,0.000571,0.000571,0.126129,0.290137,0.000000,0.422557,0.132632,0.021108,9.252711e-04
7,8,Wajir,46,2019,781263,56773,13.8,2.624669,1.66,0.020103,0.020103,0.000977,0.550222,0.000000,0.418645,0.009366,0.000093,1.975856e-04
8,9,Mandera,24,2019,867457,25940,33.4,3.508556,-1.68,0.013467,0.013467,0.032401,0.482047,0.000000,0.440438,0.025680,0.002116,5.671588e-04
9,10,Marsabit,25,2019,459785,70944,6.5,1.871802,4.57,0.000551,0.000551,0.315609,0.081227,0.000000,0.212227,0.380300,0.000574,2.213970e-04
